# Tugas Preprocessing Dataset — Titanic (Kaggle)

**Tugas:**
1. Cari dataset (Kaggle)
2. Lakukan proses Load dataset, Preprocessing (Missing Value, Outlier, Scaling, Splitting) di notebook Jupyter
3. Kirim file Notebook Jupyter (.ipynb)

**Dataset yang digunakan:** Titanic Dataset  
Link Kaggle: https://www.kaggle.com/datasets/yasserh/titanic-dataset

> Catatan: Download file `Titanic-Dataset.csv` dari link di atas, lalu taruh di folder yang sama dengan notebook ini sebelum menjalankan cell **Load Dataset** di bawah.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

## 2. Load Dataset

Ganti nama file di bawah sesuai file yang kamu download dari Kaggle.

In [ ]:
# Load dataset hasil download dari Kaggle
df = pd.read_csv('Titanic-Dataset.csv')

df.head()

In [ ]:
# Info umum dataset
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

## 3. Preprocessing — Cek Missing Value

In [ ]:
# Cek jumlah missing value per kolom
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
pd.DataFrame({'Total Missing': missing, 'Persentase (%)': missing_percent}).sort_values('Total Missing', ascending=False)

### Menangani Missing Value

Strategi:
- **Age** (numerik) → isi dengan **median** (tahan terhadap outlier)
- **Embarked** (kategorikal) → isi dengan **modus** (nilai paling sering muncul)
- **Cabin** → terlalu banyak missing, kolom di-drop

In [ ]:
# Isi missing value Age dengan median
if 'Age' in df.columns:
    df['Age'] = df['Age'].fillna(df['Age'].median())

# Isi missing value Embarked dengan modus
if 'Embarked' in df.columns:
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Drop kolom dengan missing value terlalu banyak (Cabin) dan kolom yang tidak relevan untuk model
drop_cols = [c for c in ['Cabin', 'Name', 'Ticket', 'PassengerId'] if c in df.columns]
df = df.drop(columns=drop_cols)

print("Missing value setelah penanganan:")
print(df.isnull().sum())

## 4. Preprocessing — Deteksi & Penanganan Outlier

In [ ]:
# Visualisasi outlier dengan boxplot pada kolom numerik utama
num_cols_check = [c for c in ['Age', 'Fare'] if c in df.columns]

plt.figure(figsize=(10, 4))
for i, col in enumerate(num_cols_check):
    plt.subplot(1, len(num_cols_check), i+1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot {col}')
plt.tight_layout()
plt.show()

### Menangani Outlier dengan Metode IQR

Nilai yang berada di luar batas bawah/atas (Q1 - 1.5*IQR dan Q3 + 1.5*IQR) akan di-*cap* (dipangkas) ke nilai batas terdekat, bukan dihapus, supaya data tidak terlalu banyak berkurang.

In [ ]:
def cap_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    data[column] = np.where(data[column] > upper_bound, upper_bound, data[column])
    data[column] = np.where(data[column] < lower_bound, lower_bound, data[column])
    return data

for col in num_cols_check:
    df = cap_outliers_iqr(df, col)

print("Outlier sudah di-cap. Statistik setelah penanganan:")
df[num_cols_check].describe()

## 5. Encoding Data Kategorikal

Dilakukan sebelum scaling karena scaling hanya bekerja pada data numerik.

In [ ]:
# Label Encoding untuk kolom biner (Sex)
if 'Sex' in df.columns:
    le = LabelEncoder()
    df['Sex'] = le.fit_transform(df['Sex'])

# One-Hot Encoding untuk kolom kategorikal dengan >2 kategori (Embarked)
if 'Embarked' in df.columns:
    df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

df.head()

## 6. Preprocessing — Feature Scaling

In [ ]:
# Scaling fitur numerik menggunakan StandardScaler (mean=0, std=1)
scale_cols = [c for c in ['Age', 'Fare', 'SibSp', 'Parch'] if c in df.columns]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

df[scale_cols].describe()

## 7. Preprocessing — Splitting Data (Train/Test Split)

In [ ]:
# Pisahkan fitur (X) dan target (y)
target_col = 'Survived'
X = df.drop(columns=[target_col])
y = df[target_col]

# Split menjadi data latih (80%) dan data uji (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Jumlah data latih :", X_train.shape)
print("Jumlah data uji   :", X_test.shape)

## Kesimpulan

Notebook ini telah melakukan seluruh tahapan preprocessing sesuai tugas:
1. **Load Dataset** — membaca file dataset Titanic dari Kaggle
2. **Missing Value** — mengisi nilai kosong pada kolom Age (median) dan Embarked (modus), serta drop kolom dengan missing value tinggi
3. **Outlier** — mendeteksi dengan boxplot dan menangani dengan metode IQR (capping)
4. **Encoding** — mengubah data kategorikal menjadi numerik
5. **Scaling** — menstandarisasi fitur numerik dengan StandardScaler
6. **Splitting** — membagi data menjadi data latih dan data uji (80:20)

Dataset siap digunakan untuk tahap modeling machine learning selanjutnya.